In [3]:
from evolutionSimulation.python.neuralnetworks.nn import * 
from datasets import load_dataset
import torch
import transformers

brain = Brain()
data = load_dataset("json", data_files=r"C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\python\dataset\simple_dataset.json")
shuffled_dataset = data.shuffle()

In [20]:
import torch
import time
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim
import os

animals = {
    "lion": 1,
    "crocodile": 1,
    "dragon": 1,
    "duck": 0,
    "sheep": 0,
}

def batch(batch_size, start_index, dataset):
    truth = []
    images = [sample for sample in dataset["train"][start_index:start_index + batch_size]["image"]]
    tensor = torch.tensor(images)
    tensor = tensor.view(10, 1, 28, 28)
    for animal in dataset["train"][start_index:start_index + batch_size]["name"]:
        truth.append(animals[animal])
    return tensor.float(), truth

def train(num_img, batch_size, num_epoch, model, dataset):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.Adam(model.parameters(), lr=1e-5)
    model.to(device)
    model.train()
    best_loss = float("inf")
    total_loss = 0

    for epoch in range(num_epoch):
        epoch_loss = 0
        t0 = time.perf_counter()

        for i in range(0, num_img, batch_size):
            temp_batch = batch(batch_size, i, dataset)
            predictions = model(temp_batch[0].to(device))
            ground_truth = torch.tensor(temp_batch[1]).to(device, dtype = torch.long)
            loss_fn = nn.CrossEntropyLoss()

            loss = loss_fn(predictions, ground_truth)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            if (i / num_img * 100) % 10 == 0:
                print(f"{i / num_img * 100}% | Loss: {loss.item():.4f}")
        
        avg_loss = epoch_loss / (num_img // batch_size)
        total_loss += epoch_loss
        t1 = time.perf_counter()
        print(f"Finished Epoch {epoch} in {t1 - t0} seconds, Loss: {avg_loss:.4f}")
        try: 
            os.mkdir(r'C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\modelWeights\{}'.format(num_img))
        except FileExistsError:
            pass
        torch.save(model.state_dict(), r'C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\modelWeights\{}\model{}.pt'.format(num_img, epoch))


In [22]:
train(100, 10, 3,  brain, shuffled_dataset)

0.0% | Loss: 0.2574
10.0% | Loss: 0.0477
20.0% | Loss: 0.2435
30.0% | Loss: 0.5959
40.0% | Loss: 0.8993
50.0% | Loss: 0.0382
60.0% | Loss: 0.0675
70.0% | Loss: 0.0304
80.0% | Loss: 0.7079
90.0% | Loss: 0.1000
Finished Epoch 0 in 0.13573069999984 seconds, Loss: 0.2988
0.0% | Loss: 0.2471
10.0% | Loss: 0.0489
20.0% | Loss: 0.2472
30.0% | Loss: 0.5457
40.0% | Loss: 0.7987
50.0% | Loss: 0.0455
60.0% | Loss: 0.0479
70.0% | Loss: 0.0316
80.0% | Loss: 0.6611
90.0% | Loss: 0.0858
Finished Epoch 1 in 0.14815549999912037 seconds, Loss: 0.2760
0.0% | Loss: 0.2681
10.0% | Loss: 0.0566
20.0% | Loss: 0.2593
30.0% | Loss: 0.5027
40.0% | Loss: 0.7135
50.0% | Loss: 0.0550
60.0% | Loss: 0.0361
70.0% | Loss: 0.0350
80.0% | Loss: 0.6278
90.0% | Loss: 0.0765
Finished Epoch 2 in 0.14728740000100515 seconds, Loss: 0.2630


In [19]:
brain = Brain()
brain.load_state_dict(torch.load(r'C:\Users\allan\nvim\projects\evolutionSimulation\evolutionSimulation\modelWeights\10\model2.pt'))

<All keys matched successfully>

In [6]:
import numpy as np
test2 = np.array(shuffled_dataset["train"][200002]["image"], dtype=np.uint8 )
print(shuffled_dataset["train"][10004]["name"])

sheep


In [15]:
rand = np.random.randint(0, 10000)
print(shuffled_dataset["train"][rand]["name"])
print(brain(torch.tensor(shuffled_dataset["train"][rand]["image"]).view(1, 1, 28, 28).float()))

lion
tensor([[-3.4733,  0.5263]], grad_fn=<AddmmBackward0>)
